# Fake news Classifier

## Importing Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
#
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from joblib import dump, load

## Reading Data

In [ ]:
df = pd.read_csv('../Data/CleanedData.csv')

In [ ]:
df

## Checking Data Once Again incase Missing Values or Null Values

In [ ]:
df = df.drop(columns=['Unnamed: 0'], axis=1)

In [ ]:
print(df['label'].value_counts())

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df

## Splitting Data to Content And It's Label

In [ ]:
X = df['content']
y = df['label']

### Train Test Split on the Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=0, stratify=y, test_size=0.2)

### Vectorizing The Data

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, stop_words='english')
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

## Loading in the models

In [ ]:
models = [
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    LogisticRegression(max_iter=300, solver='liblinear', class_weight='balanced')
]


### Decalring and Initializing a best mode and best accuracy variable to capture the information

In [ ]:
best_model = None
best_accuracy = 0

### Training and evaluating the model

In [ ]:

best_model = None
best_accuracy = 0
best_model_name = ""

for model in models:
    model.fit(X_train_vect, y_train)
    y_pred = model.predict(X_test_vect)
    accuracy = accuracy_score(y_test, y_pred)
    
    model_name = type(model).__name__
    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model
        best_model_name = model_name

print(f"\n? Best model: {best_model_name} with accuracy: {best_accuracy:.4f}")

### Confusion Matrix

In [ ]:
y_pred = best_model.predict(X_test_vect)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_estimator(best_model, X_test_vect, y_test)
plt.show()
